In [1]:
# ============================================================
# JOURNAL PAPER CODE
# Step 1: Original Dataset
# Step 2: PDA Feature Generation - 8 Features
# Step 3: TM Feature Generation - 5 Features
# Step 4: Scenario I and Scenario II Construction
# ============================================================

import pandas as pd
import numpy as np
import re
import os

# ============================================================
# SECTION 1: LOAD ORIGINAL DATASET
# ============================================================

DATA_PATH = "Thyroid_Diff.csv"     # change file name if needed
TARGET_COL = "Recurred"

OUT_DIR = "dataset_steps"
os.makedirs(OUT_DIR, exist_ok=True)

df_original = pd.read_csv(DATA_PATH)
df_original.columns = df_original.columns.str.strip()

print("STEP 1: ORIGINAL DATASET")
print("Shape:", df_original.shape)
print(df_original.head())

df_original.to_csv(
    os.path.join(OUT_DIR, "Step_1_Original_Dataset.csv"),
    index=False
)

STEP 1: ORIGINAL DATASET
Shape: (383, 17)
   Age Gender Smoking Hx Smoking Hx Radiothreapy Thyroid Function  \
0   27      F      No         No              No        Euthyroid   
1   34      F      No        Yes              No        Euthyroid   
2   30      F      No         No              No        Euthyroid   
3   62      F      No         No              No        Euthyroid   
4   62      F      No         No              No        Euthyroid   

          Physical Examination Adenopathy       Pathology     Focality Risk  \
0   Single nodular goiter-left         No  Micropapillary    Uni-Focal  Low   
1          Multinodular goiter         No  Micropapillary    Uni-Focal  Low   
2  Single nodular goiter-right         No  Micropapillary    Uni-Focal  Low   
3  Single nodular goiter-right         No  Micropapillary    Uni-Focal  Low   
4          Multinodular goiter         No  Micropapillary  Multi-Focal  Low   

     T   N   M Stage       Response Recurred  
0  T1a  N0  M0     I 

In [2]:
# ============================================================
# SECTION 2: HELPER FUNCTIONS FOR CLINICAL LEVEL ENCODING
# ============================================================

def extract_number(value):
    text = str(value).strip().lower()
    numbers = re.findall(r"\d+", text)
    return int(numbers[0]) if numbers else np.nan


def encode_T(value):
    text = str(value).strip().lower()
    if "t4" in text:
        return 4
    elif "t3" in text:
        return 3
    elif "t2" in text:
        return 2
    elif "t1" in text:
        return 1
    else:
        return extract_number(value)


def encode_N(value):
    text = str(value).strip().lower()
    if "n0" in text:
        return 0
    elif "n1b" in text:
        return 2
    elif "n1a" in text:
        return 1
    elif "n1" in text:
        return 1
    else:
        return extract_number(value)


def encode_M(value):
    text = str(value).strip().lower()
    if "m0" in text:
        return 0
    elif "m1" in text:
        return 1
    else:
        return extract_number(value)


def encode_stage(value):
    text = str(value).strip().lower()
    text = text.replace("stage", "").strip()

    stage_map = {
        "i": 1,
        "ii": 2,
        "iii": 3,
        "iv": 4,
        "iva": 4,
        "ivb": 4,
        "ivc": 4
    }

    if text in stage_map:
        return stage_map[text]
    else:
        return extract_number(value)


def encode_risk(value):
    text = str(value).strip().lower()

    if "low" in text:
        return 1
    elif "intermediate" in text:
        return 2
    elif "high" in text:
        return 3
    else:
        return extract_number(value)


def encode_response(value):
    text = str(value).strip().lower()

    if "excellent" in text:
        return 0
    elif "indeterminate" in text:
        return 1
    elif "biochemical" in text:
        return 2
    elif "structural" in text:
        return 3
    else:
        return extract_number(value)

In [3]:
# ============================================================
# SECTION 3: APPLY PDA AND CREATE 8 PDA FEATURES
# ============================================================

df_pda = df_original.copy()

required_columns = ["T", "N", "M", "Stage", "Risk", "Response"]

missing_cols = [col for col in required_columns if col not in df_pda.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing required columns for PDA feature generation: {missing_cols}")


# ---------------- PDA Feature 1 ----------------
df_pda["T_level"] = df_pda["T"].apply(encode_T)

# ---------------- PDA Feature 2 ----------------
df_pda["N_level"] = df_pda["N"].apply(encode_N)

# ---------------- PDA Feature 3 ----------------
df_pda["M_level"] = df_pda["M"].apply(encode_M)

# ---------------- PDA Feature 4 ----------------
df_pda["Stage_level"] = df_pda["Stage"].apply(encode_stage)

# ---------------- PDA Feature 5 ----------------
df_pda["Risk_level"] = df_pda["Risk"].apply(encode_risk)

# ---------------- PDA Feature 6 ----------------
df_pda["Response_level"] = df_pda["Response"].apply(encode_response)

# ---------------- PDA Feature 7 ----------------
df_pda["TNM_Burden"] = (
    df_pda["T_level"] +
    df_pda["N_level"] +
    df_pda["M_level"]
)

# ---------------- PDA Feature 8 ----------------
df_pda["Hierarchy_Score"] = (
    df_pda["Stage_level"] +
    df_pda["Risk_level"] +
    df_pda["Response_level"]
)


# Fill unmapped values if any
pda_features = [
    "T_level",
    "N_level",
    "M_level",
    "Stage_level",
    "Risk_level",
    "Response_level",
    "TNM_Burden",
    "Hierarchy_Score"
]

for col in pda_features:
    df_pda[col] = df_pda[col].fillna(df_pda[col].median())


print("\nSTEP 2: DATASET AFTER PDA FEATURE GENERATION")
print("Shape:", df_pda.shape)
print("PDA Features Created:", pda_features)
print(df_pda.head())

df_pda.to_csv(
    os.path.join(OUT_DIR, "Step_2_After_PDA_8_Features.csv"),
    index=False
)


STEP 2: DATASET AFTER PDA FEATURE GENERATION
Shape: (383, 25)
PDA Features Created: ['T_level', 'N_level', 'M_level', 'Stage_level', 'Risk_level', 'Response_level', 'TNM_Burden', 'Hierarchy_Score']
   Age Gender Smoking Hx Smoking Hx Radiothreapy Thyroid Function  \
0   27      F      No         No              No        Euthyroid   
1   34      F      No        Yes              No        Euthyroid   
2   30      F      No         No              No        Euthyroid   
3   62      F      No         No              No        Euthyroid   
4   62      F      No         No              No        Euthyroid   

          Physical Examination Adenopathy       Pathology     Focality  ...  \
0   Single nodular goiter-left         No  Micropapillary    Uni-Focal  ...   
1          Multinodular goiter         No  Micropapillary    Uni-Focal  ...   
2  Single nodular goiter-right         No  Micropapillary    Uni-Focal  ...   
3  Single nodular goiter-right         No  Micropapillary    Uni-Focal

In [7]:
# ============================================================
# SECTION 4: APPLY TURING MACHINE AND CREATE 5 TM FEATURES
# ============================================================

df_tm = df_pda.copy()

# ---------------- TM Feature 1 ----------------
df_tm["Stage_Risk_Gap"] = abs(
    df_tm["Stage_level"] - df_tm["Risk_level"]
)

# ---------------- TM Feature 2 ----------------
df_tm["Progression_Flag"] = np.where(
    (df_tm["Response_level"] >= 2) |
    (df_tm["Risk_level"] >= 2) |
    (df_tm["N_level"] >= 1),
    1,
    0
)

# ---------------- TM Feature 3 ----------------
df_tm["Accept_State_Score"] = (
    (2 * df_tm["Response_level"]) +
    df_tm["Risk_level"] +
    df_tm["N_level"] +
    df_tm["M_level"]
)

# ---------------- TM Feature 4 ----------------
df_tm["Transition_Code"] = (
    df_tm["T_level"] * 1000 +
    df_tm["N_level"] * 100 +
    df_tm["M_level"] * 10 +
    df_tm["Stage_level"]
)

# ---------------- TM Feature 5 ----------------
df_tm["State_ID"] = (
    df_tm["Stage_level"] * 100 +
    df_tm["Risk_level"] * 10 +
    df_tm["Response_level"]
)

tm_features = [
    "Stage_Risk_Gap",
    "Progression_Flag",
    "Accept_State_Score",
    "Transition_Code",
    "State_ID"
]

print("\nSTEP 3: DATASET AFTER TURING MACHINE FEATURE GENERATION")
print("Shape:", df_tm.shape)
print("TM Features Created:", tm_features)
print(df_tm.head())

df_tm.to_csv(
    os.path.join(OUT_DIR, "Step_3_After_TM_5_Features.csv"),
    index=False
)


STEP 3: DATASET AFTER TURING MACHINE FEATURE GENERATION
Shape: (383, 30)
TM Features Created: ['Stage_Risk_Gap', 'Progression_Flag', 'Accept_State_Score', 'Transition_Code', 'State_ID']
   Age Gender Smoking Hx Smoking Hx Radiothreapy Thyroid Function  \
0   27      F      No         No              No        Euthyroid   
1   34      F      No        Yes              No        Euthyroid   
2   30      F      No         No              No        Euthyroid   
3   62      F      No         No              No        Euthyroid   
4   62      F      No         No              No        Euthyroid   

          Physical Examination Adenopathy       Pathology     Focality  ...  \
0   Single nodular goiter-left         No  Micropapillary    Uni-Focal  ...   
1          Multinodular goiter         No  Micropapillary    Uni-Focal  ...   
2  Single nodular goiter-right         No  Micropapillary    Uni-Focal  ...   
3  Single nodular goiter-right         No  Micropapillary    Uni-Focal  ...   
4  

In [8]:
# ============================================================
# SECTION 5: FINAL AUTOMATA-ENHANCED DATASET
# PDA 8 FEATURES + TM 5 FEATURES = 13 AUTOMATA FEATURES
# ============================================================

automata_features = pda_features + tm_features

print("\nFINAL AUTOMATA FEATURES")
print("Total Automata Features:", len(automata_features))
print(automata_features)

print("\nFINAL AUTOMATA-ENHANCED DATASET")
print("Shape:", df_tm.shape)
print(df_tm.head())

df_tm.to_csv(
    os.path.join(OUT_DIR, "Step_4_Final_Automata_Enhanced_Dataset.csv"),
    index=False
)


FINAL AUTOMATA FEATURES
Total Automata Features: 13
['T_level', 'N_level', 'M_level', 'Stage_level', 'Risk_level', 'Response_level', 'TNM_Burden', 'Hierarchy_Score', 'Stage_Risk_Gap', 'Progression_Flag', 'Accept_State_Score', 'Transition_Code', 'State_ID']

FINAL AUTOMATA-ENHANCED DATASET
Shape: (383, 30)
   Age Gender Smoking Hx Smoking Hx Radiothreapy Thyroid Function  \
0   27      F      No         No              No        Euthyroid   
1   34      F      No        Yes              No        Euthyroid   
2   30      F      No         No              No        Euthyroid   
3   62      F      No         No              No        Euthyroid   
4   62      F      No         No              No        Euthyroid   

          Physical Examination Adenopathy       Pathology     Focality  ...  \
0   Single nodular goiter-left         No  Micropapillary    Uni-Focal  ...   
1          Multinodular goiter         No  Micropapillary    Uni-Focal  ...   
2  Single nodular goiter-right         N

In [9]:
# ============================================================
# SECTION 6: SCENARIO I - ORIGINAL DATASET
# ============================================================

scenario_1_df = df_original.copy()

scenario_1_features = [
    col for col in scenario_1_df.columns
    if col != TARGET_COL
]

X_scenario_1 = scenario_1_df[scenario_1_features]
y_scenario_1 = scenario_1_df[TARGET_COL]

print("\nSCENARIO I: ORIGINAL DATASET")
print("Dataset Shape:", scenario_1_df.shape)
print("Number of Features:", len(scenario_1_features))
print("Target:", TARGET_COL)
print("Features:")
print(scenario_1_features)

scenario_1_df.to_csv(
    os.path.join(OUT_DIR, "Scenario_1_Original_Dataset.csv"),
    index=False
)


SCENARIO I: ORIGINAL DATASET
Dataset Shape: (383, 17)
Number of Features: 16
Target: Recurred
Features:
['Age', 'Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Thyroid Function', 'Physical Examination', 'Adenopathy', 'Pathology', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Response']


In [11]:
# ============================================================
# SECTION 7: SCENARIO II - AUTOMATA-ENHANCED DATASET
# ============================================================

scenario_2_df = df_tm.copy()

scenario_2_features = [
    col for col in scenario_2_df.columns
    if col != TARGET_COL
]

X_scenario_2 = scenario_2_df[scenario_2_features]
y_scenario_2 = scenario_2_df[TARGET_COL]

print("\nSCENARIO II: AUTOMATA-ENHANCED DATASET")
print("Dataset Shape:", scenario_2_df.shape)
print("Number of Features:", len(scenario_2_features))
print("Target:", TARGET_COL)
print("Features:")
print(scenario_2_features)

scenario_2_df.to_csv(
    os.path.join(OUT_DIR, "Scenario_2_Automata_Enhanced_Dataset.csv"),
    index=False
)


SCENARIO II: AUTOMATA-ENHANCED DATASET
Dataset Shape: (383, 30)
Number of Features: 29
Target: Recurred
Features:
['Age', 'Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Thyroid Function', 'Physical Examination', 'Adenopathy', 'Pathology', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Response', 'T_level', 'N_level', 'M_level', 'Stage_level', 'Risk_level', 'Response_level', 'TNM_Burden', 'Hierarchy_Score', 'Stage_Risk_Gap', 'Progression_Flag', 'Accept_State_Score', 'Transition_Code', 'State_ID']


In [13]:
# ============================================================
# SECTION 8: DATASET TRANSFORMATION SUMMARY
# ============================================================

summary_table = pd.DataFrame({
    "Step": [
        "Original Dataset",
        "After PDA Feature Generation",
        "After TM Feature Generation",
        "Scenario I",
        "Scenario II"
    ],
    "Dataset Description": [
        "Original clinical thyroid recurrence dataset",
        "Original dataset with 8 PDA-derived features",
        "PDA-updated dataset with 5 TM-derived features",
        "Original dataset used for baseline modelling",
        "Final automata-enhanced dataset used for proposed modelling"
    ],
    "Samples": [
        df_original.shape[0],
        df_pda.shape[0],
        df_tm.shape[0],
        scenario_1_df.shape[0],
        scenario_2_df.shape[0]
    ],
    "Total Columns": [
        df_original.shape[1],
        df_pda.shape[1],
        df_tm.shape[1],
        scenario_1_df.shape[1],
        scenario_2_df.shape[1]
    ],
    "Predictor Features": [
        df_original.shape[1] - 1,
        df_pda.shape[1] - 1,
        df_tm.shape[1] - 1,
        len(scenario_1_features),
        len(scenario_2_features)
    ],
    "New Features Added": [
        0,
        len(pda_features),
        len(tm_features),
        0,
        len(automata_features)
    ]
})

print("\nDATASET TRANSFORMATION SUMMARY")
print(summary_table)

summary_table.to_csv(
    os.path.join(OUT_DIR, "Dataset_Transformation_Summary.csv"),
    index=False
)


DATASET TRANSFORMATION SUMMARY
                           Step  \
0              Original Dataset   
1  After PDA Feature Generation   
2   After TM Feature Generation   
3                    Scenario I   
4                   Scenario II   

                                 Dataset Description  Samples  Total Columns  \
0       Original clinical thyroid recurrence dataset      383             17   
1       Original dataset with 8 PDA-derived features      383             25   
2     PDA-updated dataset with 5 TM-derived features      383             30   
3       Original dataset used for baseline modelling      383             17   
4  Final automata-enhanced dataset used for propo...      383             30   

   Predictor Features  New Features Added  
0                  16                   0  
1                  24                   8  
2                  29                   5  
3                  16                   0  
4                  29                  13  
